In [1]:
# -*- coding: utf-8 -*-
"""Untitled3.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1UrlHwWCaSI9i6A_W6JWDoDeoHZbxqtqj

# Sección nueva
"""

# -----------------------------------------------------------------------------
# 1. IMPORTACIÓN DE LIBRERÍAS
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import ipywidgets as widgets
from IPython.display import display, HTML

# -----------------------------------------------------------------------------
# 2. GENERACIÓN DEL DATASET (22 ESTACIONES RMCAB - BOGOTÁ)
# -----------------------------------------------------------------------------
np.random.seed(42)

estaciones_rmcab = {
    "Guaymaral": {"lat": 4.7830, "lon": -74.0440, "base_aqi": 32, "tipo": "Rural/Periurbana"},
    "Usaquén": {"lat": 4.7103, "lon": -74.0306, "base_aqi": 38, "tipo": "Urbana Residencial"},
    "Suba": {"lat": 4.7410, "lon": -74.0850, "base_aqi": 52, "tipo": "Urbana Residencial"},
    "Bolivia": {"lat": 4.7180, "lon": -74.1150, "base_aqi": 48, "tipo": "Rural/Periurbana"},
    "Colina": {"lat": 4.7320, "lon": -74.0650, "base_aqi": 42, "tipo": "Urbana Residencial"},
    "Las Ferias": {"lat": 4.6903, "lon": -74.0822, "base_aqi": 58, "tipo": "Urbana Tráfico"},
    "Engativá": {"lat": 4.7010, "lon": -74.1200, "base_aqi": 62, "tipo": "Urbana Residencial"},
    "Parque Simón Bolívar": {"lat": 4.6580, "lon": -74.0840, "base_aqi": 41, "tipo": "Urbana Residencial"},
    "Móvil 7ma": {"lat": 4.6640, "lon": -74.0550, "base_aqi": 56, "tipo": "Urbana Tráfico"},
    "Móvil Fontibón": {"lat": 4.6750, "lon": -74.1380, "base_aqi": 72, "tipo": "Urbana Industrial"},
    "Fontibón": {"lat": 4.6700, "lon": -74.1440, "base_aqi": 78, "tipo": "Urbana Industrial"},
    "MinAmbiente": {"lat": 4.6097, "lon": -74.0689, "base_aqi": 45, "tipo": "Urbana Centro"},
    "Centro": {"lat": 4.6020, "lon": -74.0820, "base_aqi": 64, "tipo": "Urbana Centro"},
    "Puente Aranda": {"lat": 4.6317, "lon": -74.1175, "base_aqi": 88, "tipo": "Urbana Industrial"},
    "Kennedy": {"lat": 4.6250, "lon": -74.1611, "base_aqi": 96, "tipo": "Urbana Tráfico"},
    "Bosa": {"lat": 4.6100, "lon": -74.1900, "base_aqi": 104, "tipo": "Urbana Industrial"},
    "Carvajal - Sevillana": {"lat": 4.5978, "lon": -74.1331, "base_aqi": 118, "tipo": "Urbana Industrial"},
    "Jazmín": {"lat": 4.6180, "lon": -74.1120, "base_aqi": 82, "tipo": "Urbana Residencial"},
    "Tunal": {"lat": 4.5761, "lon": -74.1302, "base_aqi": 68, "tipo": "Urbana Tráfico"},
    "San Cristóbal": {"lat": 4.5720, "lon": -74.0830, "base_aqi": 38, "tipo": "Rural/Periurbana"},
    "Ciudad Bolívar": {"lat": 4.5606, "lon": -74.1614, "base_aqi": 82, "tipo": "Urbana Industrial"},
    "Usme": {"lat": 4.4717, "lon": -74.1122, "base_aqi": 33, "tipo": "Rural/Periurbana"}
}

fechas_historicas = pd.date_range(end="2026-09-14 23:00", periods=14*24, freq="h")
registros = []

for fecha in fechas_historicas:
    hora = fecha.hour
    ciclo_diario = 1.3 if (6 <= hora <= 10 or 17 <= hora <= 21) else 0.85

    for nombre, info in estaciones_rmcab.items():
        base = info["base_aqi"] * ciclo_diario
        aqi_val = max(5, int(np.random.normal(base, 6)))

        registros.append({
            "Fecha": fecha,
            "Estación": nombre,
            "Tipo_Zona": info["tipo"],
            "Latitud": info["lat"],
            "Longitud": info["lon"],
            "AQI": aqi_val,
            "PM2.5": round(max(2.0, aqi_val * 0.85 + np.random.normal(0, 2)), 1),
            "PM10": round(max(5.0, aqi_val * 1.30 + np.random.normal(0, 4)), 1),
            "O3": round(max(1.0, aqi_val * 0.35 + np.random.normal(0, 2)), 1),
            "NO2": round(max(2.0, aqi_val * 0.45 + np.random.normal(0, 2)), 1),
            "SO2": round(max(0.5, aqi_val * 0.08 + np.random.normal(0, 0.5)), 1),
            "CO": round(max(0.1, aqi_val * 0.04 + np.random.normal(0, 0.1)), 2)
        })

df_historico = pd.DataFrame(registros)

# Panel técnico de sondas
estados_op = ["Operativa (En Línea)", "En Mantenimiento", "Calibración Requerida"]
panel_estado_sondas = []
for nombre in estaciones_rmcab.keys():
    st = np.random.choice(estados_op, p=[0.90, 0.05, 0.05])
    panel_estado_sondas.append({
        "Estación": nombre,
        "Estado": st,
        "Batería/Energía": f"{np.random.randint(85, 100)}%",
        "Señal 4G": f"{np.random.randint(75, 100)}%",
        "Último Ping": "Hace 2 min" if st == "Operativa (En Línea)" else "Hace 40 min",
        "Salud Sensor PM2.5": "Óptima" if st == "Operativa (En Línea)" else "Revisión"
    })
df_sondas_estado = pd.DataFrame(panel_estado_sondas)

# -----------------------------------------------------------------------------
# 3. FUNCIONES AUXILIARES DE FORMATO Y COLOR
# -----------------------------------------------------------------------------
def clasificar_aqi_color(aqi):
    if aqi <= 50: return "#27ae60"
    elif aqi <= 100: return "#f39c12"
    elif aqi <= 150: return "#e67e22"
    elif aqi <= 200: return "#e74c3c"
    else: return "#8e44ad"

def estilar_aqi_celda(val):
    color = clasificar_aqi_color(val)
    texto_color = "black" if color == "#f39c12" else "white"
    return f'background-color: {color}; color: {texto_color}; font-weight: bold;'

def estilar_variacion(val):
    if "▲" in str(val): return 'color: #c0392b; font-weight: bold;'
    elif "▼" in str(val): return 'color: #27ae60; font-weight: bold;'
    return 'color: #7f8c8d;'

# -----------------------------------------------------------------------------
# 4. CONFIGURACIÓN DE PESTAÑAS (DASHBOARD)
# -----------------------------------------------------------------------------

# --- PESTAÑA 1: MAPA DE CALOR Y DISPERSIÓN (FOLIUM) ---
out_p1 = widgets.Output()
sel_contam_mapa = widgets.Dropdown(
    options=["AQI", "PM2.5", "PM10", "O3", "NO2", "SO2", "CO"],
    value="PM2.5", description="Contaminante:"
)

def render_p1(change=None):
    with out_p1:
        out_p1.clear_output(wait=True)
        df_ult = df_historico[df_historico["Fecha"] == df_historico["Fecha"].max()]
        mapa = folium.Map(location=[4.63, -74.11], zoom_start=11, tiles="CartoDB positron")

        datos_calor = df_ult[["Latitud", "Longitud", sel_contam_mapa.value]].values.tolist()
        HeatMap(datos_calor, radius=25, blur=18, min_opacity=0.4).add_to(mapa)

        for _, r in df_ult.iterrows():
            val = r[sel_contam_mapa.value]
            color = clasificar_aqi_color(r["AQI"])
            folium.CircleMarker(
                location=[r["Latitud"], r["Longitud"]],
                radius=8,
                color=color,
                fill=True,
                fill_opacity=0.8,
                popup=f"<b>{r['Estación']}</b><br>{sel_contam_mapa.value}: {val}",
                tooltip=f"{r['Estación']} ({sel_contam_mapa.value}: {val})"
            ).add_to(mapa)
        display(mapa)

sel_contam_mapa.observe(render_p1, names="value")


# --- PESTAÑA 2: TENDENCIAS 24H AVANZADAS (ANÁLISIS ESTADÍSTICO + MATRIZ HORARIA) ---
out_p2 = widgets.Output()
sel_sondas_p2 = widgets.SelectMultiple(
    options=list(estaciones_rmcab.keys()),
    value=["Carvajal - Sevillana", "Guaymaral", "Kennedy", "Puente Aranda"],
    description="Estaciones:"
)
sel_contam_p2 = widgets.Dropdown(
    options=["AQI", "PM2.5", "PM10", "O3", "NO2", "SO2", "CO"],
    value="PM2.5",
    description="Variable:"
)

def render_p2(change=None):
    with out_p2:
        out_p2.clear_output(wait=True)
        if not sel_sondas_p2.value:
            print("Selecciona al menos una estación para analizar tendencias.")
            return

        var = sel_contam_p2.value
        df_sub = df_historico[
            (df_historico["Estación"].isin(sel_sondas_p2.value)) &
            (df_historico["Fecha"] >= df_historico["Fecha"].max() - pd.Timedelta(hours=23))
        ].copy()

        # 1. TABLA RESUMEN DE TENDENCIA Y PICOS 24H
        resumen_analitico = []
        for est in sel_sondas_p2.value:
            df_e = df_sub[df_sub["Estación"] == est].sort_values("Fecha")

            val_actual = df_e[var].iloc[-1]
            val_hace_24h = df_e[var].iloc[0]
            diff = round(val_actual - val_hace_24h, 1)
            pct_change = round(((val_actual - val_hace_24h) / max(0.1, val_hace_24h)) * 100, 1)

            if diff > 0:
                indicador = f"▲ +{diff} (+{pct_change}%)"
            elif diff < 0:
                indicador = f"▼ {diff} ({pct_change}%)"
            else:
                indicador = "➖ 0.0 (0.0%)"

            idx_max = df_e[var].idxmax()
            idx_min = df_e[var].idxmin()

            pico_val = df_e.loc[idx_max, var]
            pico_hora = df_e.loc[idx_max, "Fecha"].strftime("%H:00")
            valle_val = df_e.loc[idx_min, var]
            valle_hora = df_e.loc[idx_min, "Fecha"].strftime("%H:00")

            resumen_analitico.append({
                "Estación": est,
                f"Actual ({var})": val_actual,
                "Promedio 24h": round(df_e[var].mean(), 1),
                "Tendencia (vs 24h)": indicador,
                "Hora Pico": f"{pico_hora} ({pico_val})",
                "Hora Valle": f"{valle_hora} ({valle_val})"
            })

        df_resumen = pd.DataFrame(resumen_analitico)
        display(HTML(f"<h3>📈 Resumen Ejecutivo de Tendencias 24h — [{var}]</h3>"))

        if var == "AQI":
            st_res = df_resumen.style.map(estilar_aqi_celda, subset=["Actual (AQI)"])\
                                    .map(estilar_variacion, subset=["Tendencia (vs 24h)"])\
                                    .set_properties(**{'text-align': 'center'})
        else:
            st_res = df_resumen.style.map(estilar_variacion, subset=["Tendencia (vs 24h)"])\
                                    .set_properties(**{'text-align': 'center'})
        display(st_res)

        # 2. MATRIZ HORARIA DESGLOSADA (ÚLTIMAS 12 HORAS)
        display(HTML(f"<h4>⏱️ Comportamiento Hora a Hora (Últimas 12 Horas) — {var}</h4>"))
        df_12h = df_sub[df_sub["Fecha"] >= df_sub["Fecha"].max() - pd.Timedelta(hours=11)].copy()
        df_12h["Hora"] = df_12h["Fecha"].dt.strftime("%H:00")

        tabla_pivote = df_12h.pivot(index="Estación", columns="Hora", values=var)

        if var == "AQI":
            st_piv = tabla_pivote.style.map(estilar_aqi_celda).set_properties(**{'text-align': 'center'})
        else:
            st_piv = tabla_pivote.style.background_gradient(cmap="YlOrRd", axis=1)\
                                      .set_properties(**{'text-align': 'center'})
        display(st_piv)

sel_sondas_p2.observe(render_p2, names="value")
sel_contam_p2.observe(render_p2, names="value")


# --- PESTAÑA 3: TABLA DE PRONÓSTICO A 24 HORAS ---
out_p3 = widgets.Output()
sel_sonda_p3 = widgets.Dropdown(
    options=list(estaciones_rmcab.keys()),
    value="Carvajal - Sevillana",
    description="Estación:"
)

def render_p3(change=None):
    with out_p3:
        out_p3.clear_output(wait=True)
        df_est = df_historico[df_historico["Estación"] == sel_sonda_p3.value].sort_values("Fecha")
        ultima_fecha = df_est["Fecha"].max()
        fechas_futuras = pd.date_range(start=ultima_fecha + pd.Timedelta(hours=1), periods=24, freq="h")

        media_historica_hora = df_est.groupby(df_est["Fecha"].dt.hour)["AQI"].mean()
        promedio_reciente = df_est.tail(12)["AQI"].mean()

        filas_pronostico = []
        for f in fechas_futuras:
            patron_hora = media_historica_hora[f.hour]
            val_pred = int(max(5, (promedio_reciente * 0.35) + (patron_hora * 0.65) + np.random.normal(0, 1.5)))

            if val_pred <= 50: riesgo = "Favorable (Bajo)"
            elif val_pred <= 100: riesgo = "Moderado"
            elif val_pred <= 150: riesgo = "Dañino a Grupos Sensibles"
            else: riesgo = "Dañino para la Salud"

            filas_pronostico.append({
                "Fecha / Hora Proyectada": f.strftime("%Y-%m-%d %H:00"),
                "AQI Estimado": val_pred,
                "Categoría de Riesgo": riesgo
            })

        df_p = pd.DataFrame(filas_pronostico)
        display(HTML(f"<h3>🔮 Tabla de Pronóstico AQI (Próximas 24 Horas): {sel_sonda_p3.value}</h3>"))
        st_p = df_p.style.map(estilar_aqi_celda, subset=["AQI Estimado"])\
                         .set_properties(**{'text-align': 'center'})
        display(st_p)

sel_sonda_p3.observe(render_p3, names="value")


# --- PESTAÑA 4: TABLA DE ESTADO TÉCNICO DE SONDAS ---
out_p4 = widgets.Output()

def render_p4():
    with out_p4:
        out_p4.clear_output(wait=True)
        display(HTML("<h3>🛠️ Estado Técnico de la Red de Sondas RMCAB</h3>"))

        def estilo_estado(val):
            if "Operativa" in val: return 'background-color: #d4edda; color: #155724; font-weight: bold;'
            elif "Mantenimiento" in val: return 'background-color: #fff3cd; color: #856404; font-weight: bold;'
            else: return 'background-color: #f8d7da; color: #721c24; font-weight: bold;'

        styled_df = df_sondas_estado.style.map(estilo_estado, subset=["Estado"])\
                                         .set_properties(**{'text-align': 'center'})
        display(styled_df)

# -----------------------------------------------------------------------------
# 5. INTEGRACIÓN GENERAL Y DISPLAY
# -----------------------------------------------------------------------------
tab_dashboard = widgets.Tab(children=[
    widgets.VBox([sel_contam_mapa, out_p1]),
    widgets.VBox([widgets.HBox([sel_sondas_p2, sel_contam_p2]), out_p2]),
    widgets.VBox([sel_sonda_p3, out_p3]),
    out_p4
])

tab_dashboard.set_title(0, "🔥 Mapa de Calor y Dispersión")
tab_dashboard.set_title(1, "📈 Tendencias 24h Avanzadas")
tab_dashboard.set_title(2, "🔮 Pronóstico 24h")
tab_dashboard.set_title(3, "🛠️ Estado de Sondas")

display(tab_dashboard)

# Carga inicial de todas las pestañas
render_p1()
render_p2()
render_p3()
render_p4()

In [ ]:
# -*- coding: utf-8 -*-
"""Mapa_RMCAB_Bogota_Executive.ipynb"""

import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64
import datetime

# -----------------------------------------------------------------------------
# 1. GENERACIÓN DE DATOS (22 ESTACIONES RMCAB - BOGOTÁ)
# -----------------------------------------------------------------------------
np.random.seed(42)

estaciones_rmcab = {
    "Guaymaral": {"lat": 4.7830, "lon": -74.0440, "base": 32, "tipo": "Rural/Periurbana"},
    "Usaquén": {"lat": 4.7103, "lon": -74.0306, "base": 38, "tipo": "Urbana Residencial"},
    "Suba": {"lat": 4.7410, "lon": -74.0850, "base": 52, "tipo": "Urbana Residencial"},
    "Bolivia": {"lat": 4.7180, "lon": -74.1150, "base": 48, "tipo": "Rural/Periurbana"},
    "Colina": {"lat": 4.7320, "lon": -74.0650, "base": 42, "tipo": "Urbana Residencial"},
    "Las Ferias": {"lat": 4.6903, "lon": -74.0822, "base": 58, "tipo": "Urbana Tráfico"},
    "Engativá": {"lat": 4.7010, "lon": -74.1200, "base": 62, "tipo": "Urbana Residencial"},
    "Parque Simón Bolívar": {"lat": 4.6580, "lon": -74.0840, "base": 41, "tipo": "Urbana Residencial"},
    "Móvil 7ma": {"lat": 4.6640, "lon": -74.0550, "base": 56, "tipo": "Urbana Tráfico"},
    "Móvil Fontibón": {"lat": 4.6750, "lon": -74.1380, "base": 72, "tipo": "Urbana Industrial"},
    "Fontibón": {"lat": 4.6700, "lon": -74.1440, "base": 78, "tipo": "Urbana Industrial"},
    "MinAmbiente": {"lat": 4.6097, "lon": -74.0689, "base": 45, "tipo": "Urbana Centro"},
    "Centro": {"lat": 4.6020, "lon": -74.0820, "base": 64, "tipo": "Urbana Centro"},
    "Puente Aranda": {"lat": 4.6317, "lon": -74.1175, "base": 88, "tipo": "Urbana Industrial"},
    "Kennedy": {"lat": 4.6250, "lon": -74.1611, "base": 96, "tipo": "Urbana Tráfico"},
    "Bosa": {"lat": 4.6100, "lon": -74.1900, "base": 104, "tipo": "Urbana Industrial"},
    "Carvajal - Sevillana": {"lat": 4.5978, "lon": -74.1331, "base": 118, "tipo": "Urbana Industrial"},
    "Jazmín": {"lat": 4.6180, "lon": -74.1120, "base": 82, "tipo": "Urbana Residencial"},
    "Tunal": {"lat": 4.5761, "lon": -74.1302, "base": 68, "tipo": "Urbana Tráfico"},
    "San Cristóbal": {"lat": 4.5720, "lon": -74.0830, "base": 38, "tipo": "Rural/Periurbana"},
    "Ciudad Bolívar": {"lat": 4.5606, "lon": -74.1614, "base": 82, "tipo": "Urbana Industrial"},
    "Usme": {"lat": 4.4717, "lon": -74.1122, "base": 33, "tipo": "Rural/Periurbana"}
}

registros = []
for nombre, info in estaciones_rmcab.items():
    aqi_val = max(10, int(np.random.normal(info["base"], 6)))
    registros.append({
        "Estación": nombre,
        "Tipo": info["tipo"],
        "Latitud": info["lat"],
        "Longitud": info["lon"],
        "AQI": aqi_val,
        "PM2.5": round(aqi_val * 0.85, 1),
        "PM10": round(aqi_val * 1.25, 1)
    })

df_actual = pd.DataFrame(registros)

def obtener_color(aqi):
    if aqi <= 50: return "#27ae60"
    elif aqi <= 100: return "#f39c12"
    elif aqi <= 150: return "#e67e22"
    else: return "#e74c3c"

def obtener_recomendacion(aqi):
    if aqi <= 50: return "Calidad de aire satisfactoria. Disfrute de las actividades al aire libre con normalidad."
    elif aqi <= 100: return "Calidad aceptable. Los grupos extremadamente sensibles deberían considerar limitar esfuerzos prolongados."
    elif aqi <= 150: return "Grupos sensibles pueden experimentar efectos en la salud. Se aconseja reducir actividades físicas exigentes al aire libre."
    else: return "Alerta de salud: Toda la población puede verse afectada. Evite actividades al aire libre y uso de transporte sin filtrar."

# -----------------------------------------------------------------------------
# 2. CONTROLES DE INTERFAZ EJECUTIVOS
# -----------------------------------------------------------------------------
display(HTML("<h2>🌐 Panel Ejecutivo de Calidad del Aire (Red RMCAB - Bogotá)</h2>"))
display(HTML("<p style='color: #555;'>Sistema inteligente de supervisión ambiental y análisis territorial en tiempo real.</p>"))

dropdown_variable = widgets.Dropdown(
    options=["AQI", "PM2.5", "PM10"],
    value="AQI",
    description="Contaminante:"
)

tipos_zona = ["Todas"] + list(df_actual["Tipo"].unique())
dropdown_tipo = widgets.Dropdown(
    options=tipos_zona,
    value="Todas",
    description="Tipo de Zona:"
)

btn_exportar = widgets.Button(
    description="📥 Exportar Reporte CSV",
    button_style="success",
    icon="download"
)

out_kpis = widgets.Output()
out_alerta = widgets.Output()
out_mapa = widgets.Output()
out_footer = widgets.Output()
out_descarga = widgets.Output()

def actualizar_dashboard(change=None):
    tipo_sel = dropdown_tipo.value
    var_sel = dropdown_variable.value

    df_f = df_actual if tipo_sel == "Todas" else df_actual[df_actual["Tipo"] == tipo_sel]

    # -------------------------------------------------------------------------
    # TARJETAS KPI (ESTILO MINIMALISTA PROFESIONAL)
    # -------------------------------------------------------------------------
    with out_kpis:
        clear_output(wait=True)
        promedio = round(df_f[var_sel].mean(), 1)
        max_reg = df_f.loc[df_f[var_sel].idxmax()] if not df_f.empty else None
        criticas = len(df_f[df_f["AQI"] > 100])

        kpi_html = f"""
        <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: Arial;">
            <div style="flex: 1; background: #ffffff; border: 1px solid #e0e0e0; border-top: 4px solid #2980b9; padding: 12px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 10px; color: #7f8c8d; font-weight: bold; text-transform: uppercase;">Promedio General</div>
                <div style="font-size: 20px; color: #2c3e50; font-weight: bold; margin-top: 4px;">{promedio} <span style="font-size: 11px; font-weight: normal; color: #95a5a6;">{var_sel}</span></div>
            </div>
            <div style="flex: 1; background: #ffffff; border: 1px solid #e0e0e0; border-top: 4px solid #d35400; padding: 12px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 10px; color: #7f8c8d; font-weight: bold; text-transform: uppercase;">Punto más crítico</div>
                <div style="font-size: 14px; color: #2c3e50; font-weight: bold; margin-top: 4px;">{max_reg['Estación'] if max_reg is not None else 'N/A'} <span style="color: #c0392b;">({max_reg[var_sel] if max_reg is not None else 0})</span></div>
            </div>
            <div style="flex: 1; background: #ffffff; border: 1px solid #e0e0e0; border-top: 4px solid #c0392b; padding: 12px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 10px; color: #7f8c8d; font-weight: bold; text-transform: uppercase;">Estaciones Críticas (>100 AQI)</div>
                <div style="font-size: 20px; color: #c0392b; font-weight: bold; margin-top: 4px;">{criticas} <span style="font-size: 11px; font-weight: normal; color: #95a5a6;">unidades</span></div>
            </div>
        </div>
        """
        display(HTML(kpi_html))

    # -------------------------------------------------------------------------
    # PANEL DE RECOMENDACIÓN DE SALUD INTELIGENTE
    # -------------------------------------------------------------------------
    with out_alerta:
        clear_output(wait=True)
        if not df_f.empty:
            max_aqi = df_f["AQI"].max()
            estacion_max = df_f.loc[df_f["AQI"].idxmax()]["Estación"]
            color_alerta = obtener_color(max_aqi)
            rec_texto = obtener_recomendacion(max_aqi)

            alerta_html = f"""
            <div style="background-color: #fdfefe; border: 1px solid #e0e0e0; border-left: 5px solid {color_alerta}; padding: 12px 15px; margin-bottom: 15px; border-radius: 6px; font-family: Arial; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 12px; font-weight: bold; color: #2c3e50; margin-bottom: 3px;">
                    🛡️ Protocolo Operativo y Sanitario (Pico actual en: <span style="color: {color_alerta};">{estacion_max}</span> - AQI {max_aqi})
                </div>
                <div style="font-size: 11.5px; color: #555; line-height: 1.4;">
                    {rec_texto}
                </div>
            </div>
            """
            display(HTML(alerta_html))

    # -------------------------------------------------------------------------
    # MAPA INTERACTIVO DE ALTA PRECISIÓN
    # -------------------------------------------------------------------------
    with out_mapa:
        clear_output(wait=True)
        mapa = folium.Map(location=[4.63, -74.11], zoom_start=11, tiles="CartoDB positron")

        datos_calor = df_f[["Latitud", "Longitud", var_sel]].values.tolist()
        HeatMap(datos_calor, radius=30, blur=20, min_opacity=0.4).add_to(mapa)

        for _, r in df_f.iterrows():
            color_estacion = obtener_color(r["AQI"])
            popup_html = f"""
            <div style="font-family: Arial; font-size: 12px; width: 180px; line-height: 1.4;">
                <b style="color: #2c3e50; font-size: 13px;">{r['Estación']}</b><br>
                <span style="background: #ecf0f1; padding: 2px 6px; border-radius: 4px; font-size: 10px;">{r['Tipo']}</span><br>
                <hr style="margin: 6px 0; border: none; border-top: 1px solid #bdc3c7;">
                <b>AQI:</b> <span style="color: {color_estacion}; font-weight: bold;">{r['AQI']}</span><br>
                <b>PM2.5:</b> {r['PM2.5']} µg/m³<br>
                <b>PM10:</b> {r['PM10']} µg/m³
            </div>
            """

            folium.CircleMarker(
                location=[r["Latitud"], r["Longitud"]],
                radius=10,
                color="#2c3e50",
                weight=1,
                fill=True,
                fill_color=color_estacion,
                fill_opacity=0.9,
                popup=folium.Popup(popup_html, max_width=220),
                tooltip=f"{r['Estación']} | {var_sel}: {r[var_sel]}"
            ).add_to(mapa)

        leyenda_html = """
        <div style="
            position: fixed;
            bottom: 30px; right: 30px; width: 190px;
            background-color: white; z-index: 9999; font-size: 11px;
            padding: 10px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.2);
            font-family: Arial, sans-serif;">
          <p style="margin: 0 0 5px 0; font-weight: bold; text-align: center;">Índice AQI (Riesgo)</p>
          <i style="background: #27ae60; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Favorable (0-50)<br>
          <i style="background: #f39c12; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Moderado (51-100)<br>
          <i style="background: #e67e22; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Sensibles (101-150)<br>
          <i style="background: #e74c3c; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Dañino (>150)
        </div>
        """
        mapa.get_root().html.add_child(folium.Element(leyenda_html))
        display(HTML(mapa._repr_html_()))

    # -------------------------------------------------------------------------
    # BARRA DE PIE DE PÁGINA (ESTADO DE RED Y HORA)
    # -------------------------------------------------------------------------
    with out_footer:
        clear_output(wait=True)
        tiempo_actual = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        footer_html = f"""
        <div style="display: flex; justify-content: space-between; font-family: Arial; font-size: 11px; color: #7f8c8d; margin-top: 10px; border-top: 1px solid #e0e0e0; padding-top: 6px;">
            <div>🟢 Red RMCAB Activa ({len(df_f)} estaciones analizadas)</div>
            <div>Última sincronización: {tiempo_actual}</div>
        </div>
        """
        display(HTML(footer_html))

# -----------------------------------------------------------------------------
# 3. EXPORTACIÓN CSV
# -----------------------------------------------------------------------------
def descargar_csv(b):
    with out_descarga:
        clear_output()
        tipo_sel = dropdown_tipo.value
        df_f = df_actual if tipo_sel == "Todas" else df_actual[df_actual["Tipo"] == tipo_sel]

        csv_data = df_f.to_csv(index=False)
        b64 = base64.b64encode(csv_data.encode()).decode()
        nombre_archivo = f"reporte_rmcab_{tipo_sel.lower().replace(' ', '_')}.csv"

        html_link = f"""
        <div style="margin-top: 8px; font-family: Arial; font-size: 11.5px;">
            <a download="{nombre_archivo}" href="data:text/csv;base64,{b64}" target="_blank"
               style="background: #27ae60; color: white; padding: 5px 10px; border-radius: 4px; text-decoration: none; font-weight: bold;">
               💾 Archivo listo para guardar ({nombre_archivo})
            </a>
        </div>
        """
        display(HTML(html_link))

btn_exportar.on_click(descargar_csv)

# Observadores
dropdown_variable.observe(actualizar_dashboard, names="value")
dropdown_tipo.observe(actualizar_dashboard, names="value")

# Estructura visual interactiva
display(widgets.HBox([dropdown_variable, dropdown_tipo, btn_exportar]))
display(out_kpis)
display(out_alerta)
display(out_mapa)
display(out_footer)
display(out_descarga)

# Carga inicial
actualizar_dashboard()

Output()

Output()

Output()

Output()

Output()

In [ ]:
# -*- coding: utf-8 -*-
"""Mapa_RMCAB_Bogota_Interactivo_Pro.ipynb"""

import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64
import datetime

# -----------------------------------------------------------------------------
# 1. GENERACIÓN DE DATOS (22 ESTACIONES RMCAB - BOGOTÁ)
# -----------------------------------------------------------------------------
np.random.seed(42)

estaciones_rmcab = {
    "Guaymaral": {"lat": 4.7830, "lon": -74.0440, "base": 32, "tipo": "Rural/Periurbana"},
    "Usaquén": {"lat": 4.7103, "lon": -74.0306, "base": 38, "tipo": "Urbana Residencial"},
    "Suba": {"lat": 4.7410, "lon": -74.0850, "base": 52, "tipo": "Urbana Residencial"},
    "Bolivia": {"lat": 4.7180, "lon": -74.1150, "base": 48, "tipo": "Rural/Periurbana"},
    "Colina": {"lat": 4.7320, "lon": -74.0650, "base": 42, "tipo": "Urbana Residencial"},
    "Las Ferias": {"lat": 4.6903, "lon": -74.0822, "base": 58, "tipo": "Urbana Tráfico"},
    "Engativá": {"lat": 4.7010, "lon": -74.1200, "base": 62, "tipo": "Urbana Residencial"},
    "Parque Simón Bolívar": {"lat": 4.6580, "lon": -74.0840, "base": 41, "tipo": "Urbana Residencial"},
    "Móvil 7ma": {"lat": 4.6640, "lon": -74.0550, "base": 56, "tipo": "Urbana Tráfico"},
    "Móvil Fontibón": {"lat": 4.6750, "lon": -74.1380, "base": 72, "tipo": "Urbana Industrial"},
    "Fontibón": {"lat": 4.6700, "lon": -74.1440, "base": 78, "tipo": "Urbana Industrial"},
    "MinAmbiente": {"lat": 4.6097, "lon": -74.0689, "base": 45, "tipo": "Urbana Centro"},
    "Centro": {"lat": 4.6020, "lon": -74.0820, "base": 64, "tipo": "Urbana Centro"},
    "Puente Aranda": {"lat": 4.6317, "lon": -74.1175, "base": 88, "tipo": "Urbana Industrial"},
    "Kennedy": {"lat": 4.6250, "lon": -74.1611, "base": 96, "tipo": "Urbana Tráfico"},
    "Bosa": {"lat": 4.6100, "lon": -74.1900, "base": 104, "tipo": "Urbana Industrial"},
    "Carvajal - Sevillana": {"lat": 4.5978, "lon": -74.1331, "base": 118, "tipo": "Urbana Industrial"},
    "Jazmín": {"lat": 4.6180, "lon": -74.1120, "base": 82, "tipo": "Urbana Residencial"},
    "Tunal": {"lat": 4.5761, "lon": -74.1302, "base": 68, "tipo": "Urbana Tráfico"},
    "San Cristóbal": {"lat": 4.5720, "lon": -74.0830, "base": 38, "tipo": "Rural/Periurbana"},
    "Ciudad Bolívar": {"lat": 4.5606, "lon": -74.1614, "base": 82, "tipo": "Urbana Industrial"},
    "Usme": {"lat": 4.4717, "lon": -74.1122, "base": 33, "tipo": "Rural/Periurbana"}
}

registros = []
for nombre, info in estaciones_rmcab.items():
    aqi_val = max(10, int(np.random.normal(info["base"], 6)))
    registros.append({
        "Estación": nombre,
        "Tipo": info["tipo"],
        "Latitud": info["lat"],
        "Longitud": info["lon"],
        "AQI": aqi_val,
        "PM2.5": round(aqi_val * 0.85, 1),
        "PM10": round(aqi_val * 1.25, 1)
    })

df_actual = pd.DataFrame(registros)

def obtener_color(aqi):
    if aqi <= 50: return "#27ae60"
    elif aqi <= 100: return "#f39c12"
    elif aqi <= 150: return "#e67e22"
    else: return "#e74c3c"

def obtener_estado(aqi):
    if aqi <= 50: return "Favorable (Verde)"
    elif aqi <= 100: return "Moderado (Amarillo)"
    elif aqi <= 150: return "Dañino para Grupos Sensibles (Naranja)"
    else: return "Dañino a la Salud (Rojo)"

def obtener_recomendacion(aqi):
    if aqi <= 50: return "Calidad de aire óptima. Actividades al aire libre sin restricciones."
    elif aqi <= 100: return "Calidad aceptable. Personas extremadamente sensibles deben moderar esfuerzos prolongados."
    elif aqi <= 150: return "Riesgo para niños, adultos mayores y personas con afecciones respiratorias. Reducir ejercicio intenso."
    else: return "Alerta general: Evite por completo la actividad física al aire libre y uso de transportes abiertos."

# -----------------------------------------------------------------------------
# 2. CONTROLES DE LA INTERFAZ
# -----------------------------------------------------------------------------
display(HTML("<h2>🌐 Panel de Monitoreo RMCAB Bogotá - Interactivo</h2>"))
display(HTML("<p style='color: #555;'>Explore las estaciones, consulte sus reportes detallados y controle la visualización térmica.</p>"))

dropdown_variable = widgets.Dropdown(
    options=["AQI", "PM2.5", "PM10"],
    value="AQI",
    description="Contaminante:"
)

tipos_zona = ["Todas"] + list(df_actual["Tipo"].unique())
dropdown_tipo = widgets.Dropdown(
    options=tipos_zona,
    value="Todas",
    description="Tipo de Zona:"
)

# Botón para encender/apagar el mapa de calor
btn_toggle_calor = widgets.ToggleButton(
    value=True,
    description="🔥 Ocultar Mapa de Calor",
    button_style="info",
    tooltip="Permite poner o quitar la capa de calor del mapa"
)

btn_exportar = widgets.Button(
    description="📥 Exportar CSV",
    button_style="success",
    icon="download"
)

out_kpis = widgets.Output()
out_alerta = widgets.Output()
out_mapa = widgets.Output()
out_footer = widgets.Output()
out_descarga = widgets.Output()

def actualizar_dashboard(change=None):
    tipo_sel = dropdown_tipo.value
    var_sel = dropdown_variable.value
    mostrar_calor = btn_toggle_calor.value

    # Cambiar texto del botón según su estado
    if mostrar_calor:
        btn_toggle_calor.description = "🔥 Ocultar Mapa de Calor"
    else:
        btn_toggle_calor.description = "🔥 Mostrar Mapa de Calor"

    df_f = df_actual if tipo_sel == "Todas" else df_actual[df_actual["Tipo"] == tipo_sel]

    # -------------------------------------------------------------------------
    # TARJETAS KPI
    # -------------------------------------------------------------------------
    with out_kpis:
        clear_output(wait=True)
        promedio = round(df_f[var_sel].mean(), 1)
        max_reg = df_f.loc[df_f[var_sel].idxmax()] if not df_f.empty else None
        criticas = len(df_f[df_f["AQI"] > 100])

        kpi_html = f"""
        <div style="display: flex; gap: 12px; margin-bottom: 12px; font-family: Arial;">
            <div style="flex: 1; background: #ffffff; border: 1px solid #e0e0e0; border-top: 4px solid #2980b9; padding: 12px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 10px; color: #7f8c8d; font-weight: bold; text-transform: uppercase;">Promedio General</div>
                <div style="font-size: 20px; color: #2c3e50; font-weight: bold; margin-top: 4px;">{promedio} <span style="font-size: 11px; font-weight: normal; color: #95a5a6;">{var_sel}</span></div>
            </div>
            <div style="flex: 1; background: #ffffff; border: 1px solid #e0e0e0; border-top: 4px solid #d35400; padding: 12px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 10px; color: #7f8c8d; font-weight: bold; text-transform: uppercase;">Punto más crítico</div>
                <div style="font-size: 14px; color: #2c3e50; font-weight: bold; margin-top: 4px;">{max_reg['Estación'] if max_reg is not None else 'N/A'} <span style="color: #c0392b;">({max_reg[var_sel] if max_reg is not None else 0})</span></div>
            </div>
            <div style="flex: 1; background: #ffffff; border: 1px solid #e0e0e0; border-top: 4px solid #c0392b; padding: 12px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 10px; color: #7f8c8d; font-weight: bold; text-transform: uppercase;">Estaciones Críticas (>100 AQI)</div>
                <div style="font-size: 20px; color: #c0392b; font-weight: bold; margin-top: 4px;">{criticas} <span style="font-size: 11px; font-weight: normal; color: #95a5a6;">unidades</span></div>
            </div>
        </div>
        """
        display(HTML(kpi_html))

    # -------------------------------------------------------------------------
    # PANEL DE ALERTA DINÁMICA
    # -------------------------------------------------------------------------
    with out_alerta:
        clear_output(wait=True)
        if not df_f.empty:
            max_aqi = df_f["AQI"].max()
            estacion_max = df_f.loc[df_f["AQI"].idxmax()]["Estación"]
            color_alerta = obtener_color(max_aqi)
            rec_texto = obtener_recomendacion(max_aqi)

            alerta_html = f"""
            <div style="background-color: #fdfefe; border: 1px solid #e0e0e0; border-left: 5px solid {color_alerta}; padding: 12px 15px; margin-bottom: 15px; border-radius: 6px; font-family: Arial; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
                <div style="font-size: 12px; font-weight: bold; color: #2c3e50; margin-bottom: 3px;">
                    🛡️ Protocolo Sanitario Activo (Pico actual en: <span style="color: {color_alerta};">{estacion_max}</span> - AQI {max_aqi})
                </div>
                <div style="font-size: 11.5px; color: #555; line-height: 1.4;">
                    {rec_texto}
                </div>
            </div>
            """
            display(HTML(alerta_html))

    # -------------------------------------------------------------------------
    # MAPA CON POPUPS ENRIQUECIDOS Y CAPA TÉRMICA OPCIONAL
    # -------------------------------------------------------------------------
    with out_mapa:
        clear_output(wait=True)
        mapa = folium.Map(location=[4.63, -74.11], zoom_start=11, tiles="CartoDB positron")

        # Añadir o ignorar capa de calor según el estado del botón
        if mostrar_calor:
            datos_calor = df_f[["Latitud", "Longitud", var_sel]].values.tolist()
            HeatMap(datos_calor, radius=30, blur=20, min_opacity=0.4).add_to(mapa)

        # Marcadores con información detallada al tocarlos
        for _, r in df_f.iterrows():
            color_estacion = obtener_color(r["AQI"])
            estado_texto = obtener_estado(r["AQI"])
            recomendacion_estacion = obtener_recomendacion(r["AQI"])

            popup_html = f"""
            <div style="font-family: Arial; font-size: 12px; width: 230px; line-height: 1.5; padding: 4px;">
                <b style="color: #2c3e50; font-size: 14px; border-bottom: 2px solid {color_estacion}; display: block; padding-bottom: 3px;">{r['Estación']}</b>
                <span style="background: #ecf0f1; padding: 2px 6px; border-radius: 4px; font-size: 10.5px; display: inline-block; margin-top: 5px;">Tipo: {r['Tipo']}</span><br>

                <div style="margin-top: 8px; background: #f9f9f9; padding: 6px; border-radius: 4px; border: 1px solid #eee;">
                    <b>Estado:</b> <span style="color: {color_estacion}; font-weight: bold;">{estado_texto}</span><br>
                    <b>AQI General:</b> {r['AQI']}<br>
                    <b>PM2.5:</b> {r['PM2.5']} µg/m³<br>
                    <b>PM10:</b> {r['PM10']} µg/m³
                </div>

                <div style="margin-top: 6px; font-size: 11px; color: #444;">
                    <b>Recomendación operativa:</b><br>
                    <i>{recomendacion_estacion}</i>
                </div>

                <div style="margin-top: 6px; font-size: 10px; color: #888; text-align: right;">
                    Lat: {r['Latitud']} | Lon: {r['Longitud']}
                </div>
            </div>
            """

            folium.CircleMarker(
                location=[r["Latitud"], r["Longitud"]],
                radius=10,
                color="#2c3e50",
                weight=1,
                fill=True,
                fill_color=color_estacion,
                fill_opacity=0.9,
                popup=folium.Popup(popup_html, max_width=260),
                tooltip=f"Tocar para ver reporte: {r['Estación']} ({var_sel}: {r[var_sel]})"
            ).add_to(mapa)

        leyenda_html = """
        <div style="
            position: fixed;
            bottom: 30px; right: 30px; width: 190px;
            background-color: white; z-index: 9999; font-size: 11px;
            padding: 10px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.2);
            font-family: Arial, sans-serif;">
          <p style="margin: 0 0 5px 0; font-weight: bold; text-align: center;">Índice AQI (Riesgo)</p>
          <i style="background: #27ae60; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Favorable (0-50)<br>
          <i style="background: #f39c12; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Moderado (51-100)<br>
          <i style="background: #e67e22; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Sensibles (101-150)<br>
          <i style="background: #e74c3c; width: 12px; height: 12px; display: inline-block; margin-right: 5px;"></i> Dañino (>150)
        </div>
        """
        mapa.get_root().html.add_child(folium.Element(leyenda_html))
        display(HTML(mapa._repr_html_()))

    # -------------------------------------------------------------------------
    # FOOTER DE ESTADO
    # -------------------------------------------------------------------------
    with out_footer:
        clear_output(wait=True)
        tiempo_actual = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        footer_html = f"""
        <div style="display: flex; justify-content: space-between; font-family: Arial; font-size: 11px; color: #7f8c8d; margin-top: 10px; border-top: 1px solid #e0e0e0; padding-top: 6px;">
            <div>🟢 Red RMCAB Activa ({len(df_f)} estaciones filtradas)</div>
            <div>Última sincronización: {tiempo_actual}</div>
        </div>
        """
        display(HTML(footer_html))

# -----------------------------------------------------------------------------
# 3. EXPORTACIÓN CSV
# -----------------------------------------------------------------------------
def descargar_csv(b):
    with out_descarga:
        clear_output()
        tipo_sel = dropdown_tipo.value
        df_f = df_actual if tipo_sel == "Todas" else df_actual[df_actual["Tipo"] == tipo_sel]

        csv_data = df_f.to_csv(index=False)
        b64 = base64.b64encode(csv_data.encode()).decode()
        nombre_archivo = f"reporte_rmcab_{tipo_sel.lower().replace(' ', '_')}.csv"

        html_link = f"""
        <div style="margin-top: 8px; font-family: Arial; font-size: 11.5px;">
            <a download="{nombre_archivo}" href="data:text/csv;base64,{b64}" target="_blank"
               style="background: #27ae60; color: white; padding: 5px 10px; border-radius: 4px; text-decoration: none; font-weight: bold;">
               💾 Archivo listo para guardar ({nombre_archivo})
            </a>
        </div>
        """
        display(HTML(html_link))

btn_exportar.on_click(descargar_csv)

# Conectar observadores a todos los controles (incluyendo el botón del mapa de calor)
dropdown_variable.observe(actualizar_dashboard, names="value")
dropdown_tipo.observe(actualizar_dashboard, names="value")
btn_toggle_calor.observe(actualizar_dashboard, names="value")

# Disposición visual de los controles
display(widgets.HBox([dropdown_variable, dropdown_tipo, btn_toggle_calor, btn_exportar]))
display(out_kpis)
display(out_alerta)
display(out_mapa)
display(out_footer)
display(out_descarga)

# Carga inicial
actualizar_dashboard()

Output()

Output()

Output()

Output()

Output()